# System Information — HPC Cluster Environment Discovery

**Purpose:** establish a reproducible, evidence-based description of the HPC environment before designing features or training machine-learning models.

This notebook is an **environment inventory and baseline collection tool**, not a modelling notebook. It is intended to run on the cluster's **headnode** and, where passwordless SSH is available, collect the same read-only inventory from each compute node.

## Questions this inventory should help answer

1. What hardware and operating-system resources are actually available on each node (CPU topology, memory, NUMA, storage, network, accelerators and sensors)?
2. How consistent are the compute nodes, and which differences are intentional?
3. Which HPC software components are installed (Slurm, MPI, compilers, libraries, containers and Python packages)?
4. Which storage and interconnect technologies are present (for example NFS, local filesystems, RDMA or InfiniBand)?
5. Which monitoring and telemetry facilities are available (Node Exporter, Prometheus, PSI, thermal sensors, time synchronisation)?
6. Which environmental constraints may affect data collection, feature engineering, model execution or deployment?

## Collection and safety notes

- The commands are **read-only**. The notebook does not install packages, change services, alter configuration, submit jobs, or modify cluster settings.
- It does not use `sudo`, collect environment variables, read credentials, dump configuration files, or inspect users' running job contents.
- Remote collection uses SSH in `BatchMode` (no interactive password prompt). If SSH is unavailable, that node is recorded as unreachable; the notebook does not try alternate credentials.
- Some details (for example DMI/BIOS data, fan sensors or privileged device information) may not be readable by an unprivileged account. Such gaps are recorded, not inferred.
- Run this on the headnode when the cluster is in a representative, ordinary operating state. Record whether benchmark or fault-injection workloads were active when the inventory was taken.

**Important:** a one-time system inventory describes the environment at collection time. It does not replace time-series telemetry or prove that a metric was continuously available during the historical dataset period.

## 1. Collection configuration

The initial topology below reflects the cluster layout established during our Node Exporter work. Verify the addresses and roles before running. The notebook uses the current Linux username for SSH; change `SSH_USER` only if the cluster account differs.

The Prometheus URL defaults to the local API on the headnode. If Prometheus is hosted elsewhere, update it here.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import getpass
import json
import os
import re
import socket
import subprocess
import sys

CLUSTER_NODES = [
    {"name": "headnode",   "address": "10.100.0.240", "role": "headnode / management / monitoring"},
    {"name": "compute001", "address": "10.100.0.198", "role": "compute"},
    {"name": "compute002", "address": "10.100.0.167", "role": "compute"},
    {"name": "compute003", "address": "10.100.0.66",  "role": "compute"},
]

# None means use the actual local hostname. Set explicitly if the headnode's
# hostname differs from the inventory name above.
LOCAL_NODE_NAME_OVERRIDE = None
SSH_USER = getpass.getuser()
SSH_CONNECT_TIMEOUT_SECONDS = 8
COLLECT_REMOTE_NODES = True

NODE_EXPORTER_PORT = 9100
PROMETHEUS_URL = "http://127.0.0.1:9090"  # edit if Prometheus is hosted elsewhere
HTTP_TIMEOUT_SECONDS = 4

OUTPUT_DIR = Path.cwd() / "system_information_output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

local_short_name = (LOCAL_NODE_NAME_OVERRIDE or socket.gethostname()).split(".")[0]
collection_started_local = datetime.now().astimezone().isoformat(timespec="seconds")
collection_started_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")

print(f"Local hostname: {socket.gethostname()}")
print(f"Local node identity used for matching: {local_short_name}")
print(f"SSH user: {SSH_USER}")
print(f"Remote collection enabled: {COLLECT_REMOTE_NODES}")
print(f"Prometheus API: {PROMETHEUS_URL}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")
print("\nConfigured nodes:")
for node in CLUSTER_NODES:
    print(f"  {node['name']:<12} {node['address']:<16} {node['role']}")

Local hostname: headnode
Local node identity used for matching: headnode
SSH user: ubuntu
Remote collection enabled: True
Prometheus API: http://127.0.0.1:9090
Output directory: /home/ubuntu/system_information_output

Configured nodes:
  headnode     10.100.0.240     headnode / management / monitoring
  compute001   10.100.0.198     compute
  compute002   10.100.0.167     compute
  compute003   10.100.0.66      compute


## 2. Read-only host collector

The collector uses Python's standard library and common Linux utilities. It records command availability and return codes so that absence of a tool is distinguishable from an empty result. Outputs are capped to avoid creating an excessively large inventory file.

The same collector source is executed locally and sent over SSH to remote nodes. It gathers hardware, OS, storage, network, HPC software, scheduler, telemetry and time-synchronisation information.

In [2]:
COLLECTOR_SOURCE = r'''import os, sys, json, re, socket, platform, subprocess, shutil, pathlib, datetime, urllib.request, urllib.error

MAX_OUTPUT = 500000

def run_cmd(argv, timeout=12):
    try:
        p = subprocess.run(argv, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, timeout=timeout, check=False)
        return {"available": True, "returncode": p.returncode,
                "stdout": (p.stdout or "")[:MAX_OUTPUT],
                "stderr": (p.stderr or "")[:12000]}
    except FileNotFoundError:
        return {"available": False, "returncode": None, "stdout": "", "stderr": "command not found"}
    except subprocess.TimeoutExpired as e:
        out = e.stdout.decode(errors="replace") if isinstance(e.stdout, bytes) else (e.stdout or "")
        err = e.stderr.decode(errors="replace") if isinstance(e.stderr, bytes) else (e.stderr or "")
        return {"available": True, "returncode": None, "stdout": out[:MAX_OUTPUT], "stderr": (err + "\nTIMEOUT")[:12000]}
    except Exception as e:
        return {"available": False, "returncode": None, "stdout": "", "stderr": f"{type(e).__name__}: {e}"}

def read_text(path, limit=30000):
    try:
        return pathlib.Path(path).read_text(errors="replace")[:limit]
    except Exception as e:
        return f"UNAVAILABLE: {type(e).__name__}: {e}"

def http_text(url, timeout=4):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "hpc-system-inventory/1.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return {"ok": True, "status": r.status, "body": r.read(8_000_000).decode("utf-8", "replace")}
    except Exception as e:
        return {"ok": False, "error": f"{type(e).__name__}: {e}"}

def parse_prometheus_summary(body):
    result = {"sample_lines": 0, "build_info": [], "collector_success": {},
              "systemd_unit_state_series": 0, "metric_families": []}
    family_names = set()
    for line in body.splitlines():
        if not line or line.startswith("#"):
            continue
        result["sample_lines"] += 1
        metric = line.split("{", 1)[0].split(" ", 1)[0]
        family_names.add(metric)
        if metric == "node_exporter_build_info":
            result["build_info"].append(line[:1500])
        elif metric == "node_scrape_collector_success":
            m = re.search(r'collector="([^"]+)"', line)
            if m:
                try: result["collector_success"][m.group(1)] = float(line.rsplit(" ", 1)[1])
                except Exception: pass
        elif metric == "node_systemd_unit_state":
            result["systemd_unit_state_series"] += 1
    result["metric_families"] = sorted(family_names)
    result["metric_family_count"] = len(family_names)
    return result

now = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec="seconds")
identity = {
    "hostname": socket.gethostname(),
    "fqdn": socket.getfqdn(),
    "collection_time_utc": now,
    "os_name": platform.system(),
    "os_release_kernel": platform.release(),
    "architecture": platform.machine(),
    "platform": platform.platform(),
    "python_version": platform.python_version(),
    "python_executable": sys.executable,
    "effective_uid": (os.geteuid() if hasattr(os, "geteuid") else None),
}

commands = {
    "uname": ["uname", "-a"],
    "lscpu": ["lscpu"],
    "lscpu_json": ["lscpu", "-J"],
    "nproc": ["nproc"],
    "numactl_hardware": ["numactl", "--hardware"],
    "lstopo": ["lstopo-no-graphics"],
    "free_bytes": ["free", "-b"],
    "swapon": ["swapon", "--show", "--bytes"],
    "lsmem": ["lsmem", "--json"],
    "lsblk_json": ["lsblk", "-J", "-o", "NAME,KNAME,TYPE,SIZE,ROTA,RO,TRAN,FSTYPE,MOUNTPOINTS,MODEL,SERIAL"],
    "df_bytes": ["df", "-B1", "-T", "-x", "tmpfs", "-x", "devtmpfs"],
    "findmnt_json": ["findmnt", "-J", "-o", "TARGET,SOURCE,FSTYPE,OPTIONS,SIZE,AVAIL,USE%"],
    "ip_addr_json": ["ip", "-j", "-details", "addr"],
    "ip_route_json": ["ip", "-j", "route"],
    "ip_link_stats_json": ["ip", "-s", "-j", "link"],
    "socket_summary": ["ss", "-s"],
    "slurm_sinfo": ["sinfo", "-N", "-h", "-o", "%N|%P|%t|%c|%m|%e|%f"],
    "slurm_nodes": ["scontrol", "show", "nodes", "-o"],
    "slurm_version": ["srun", "--version"],
    "mpi_compiler": ["mpicc", "--showme:version"],
    "mpi_launcher": ["mpirun", "--version"],
    "mpi_info_version": ["ompi_info", "--version"],
    "gcc_version": ["gcc", "--version"],
    "gfortran_version": ["gfortran", "--version"],
    "cmake_version": ["cmake", "--version"],
    "python3_version": ["python3", "--version"],
    "apptainer_version": ["apptainer", "--version"],
    "singularity_version": ["singularity", "--version"],
    "docker_version": ["docker", "--version"],
    "podman_version": ["podman", "--version"],
    "nvidia_smi_list": ["nvidia-smi", "-L"],
    "nvidia_gpu_query": ["nvidia-smi", "--query-gpu=name,uuid,memory.total,driver_version", "--format=csv,noheader"],
    "rocm_smi": ["rocm-smi", "--showproductname"],
    "rdma_devices": ["ibv_devinfo", "-l"],
    "rdma_link": ["rdma", "link", "show"],
    "infiniband_status": ["ibstat"],
    "nfs_mounts": ["nfsstat", "-m"],
    "chrony_tracking": ["chronyc", "tracking"],
    "chrony_sources": ["chronyc", "sources", "-v"],
    "sensors_json": ["sensors", "-j"],
    "kernel_modules": ["lsmod"],
}
results = {name: run_cmd(argv, timeout=18) for name, argv in commands.items()}

# Keep package inventory targeted to the HPC and data-science stack.
pkg = run_cmd(["dpkg-query", "-W", "-f=${binary:Package}\t${Version}\n"], timeout=25)
if pkg.get("available"):
    wanted = re.compile(r"^(gcc|g\+\+|gfortran|make|cmake|python3|openmpi|libopenmpi|mpich|libmpich|slurm|munge|node-exporter|prometheus-node-exporter|prometheus|chrony|nfs-common|numactl|hwloc|environment-modules|lmod|libfabric|rdma-core|libibverbs|libpmix|ucx|libucx|hdf5|libhdf5|openblas|libopenblas|fftw|libfftw|git|apptainer|singularity|docker|podman|jupyter|ipython|numpy|pandas|scipy|scikit-learn|pyarrow|polars|xgboost|lightgbm)(:.*)?$")
    filtered = []
    for line in pkg.get("stdout", "").splitlines():
        name = line.split("\t", 1)[0]
        if wanted.search(name): filtered.append(line)
    pkg["stdout"] = "\n".join(filtered)
    pkg["matched_package_count"] = len(filtered)
results["selected_dpkg_packages"] = pkg

# Service state only; no unit files or service environment variables are read.
service_names = ["slurmctld", "slurmd", "munge", "chrony", "node-exporter",
                 "prometheus-node-exporter", "prometheus", "nfs-client.target"]
services = {}
for unit in service_names:
    services[unit] = run_cmd(["systemctl", "show", unit, "--no-pager",
                              "-p", "LoadState", "-p", "ActiveState", "-p", "SubState", "-p", "UnitFileState"], timeout=6)

# Kernel-provided telemetry and device inventory.
proc_files = {p: read_text(p) for p in [
    "/proc/meminfo", "/proc/cpuinfo", "/proc/pressure/cpu",
    "/proc/pressure/memory", "/proc/pressure/io", "/proc/diskstats",
    "/proc/net/dev", "/proc/uptime", "/etc/os-release"]}

sysfs = {}
for key, path in {
    "cpu_governors": "/sys/devices/system/cpu",
    "hwmon_devices": "/sys/class/hwmon",
    "thermal_devices": "/sys/class/thermal",
    "infiniband_devices": "/sys/class/infiniband",
    "drm_devices": "/sys/class/drm",
    "nvme_devices": "/sys/class/nvme",
}.items():
    try: sysfs[key] = sorted(os.listdir(path))
    except Exception as e: sysfs[key] = f"UNAVAILABLE: {type(e).__name__}: {e}"

# CPU frequency governors and readable hwmon labels/temperatures, without sudo.
try:
    governors = set()
    for p in pathlib.Path("/sys/devices/system/cpu").glob("cpu[0-9]*/cpufreq/scaling_governor"):
        governors.add(p.read_text().strip())
    sysfs["cpu_frequency_governors"] = sorted(governors)
except Exception as e: sysfs["cpu_frequency_governors"] = f"UNAVAILABLE: {type(e).__name__}: {e}"
try:
    hwmon_readings = []
    for hw in pathlib.Path("/sys/class/hwmon").glob("hwmon*"):
        name = (hw / "name").read_text().strip() if (hw / "name").exists() else hw.name
        for inp in hw.glob("temp*_input"):
            label_file = inp.with_name(inp.name.replace("_input", "_label"))
            label = label_file.read_text().strip() if label_file.exists() else inp.stem
            value = inp.read_text().strip()
            hwmon_readings.append({"device": name, "sensor": label, "raw_value": value, "unit": "millidegrees_Celsius"})
    sysfs["hwmon_temperature_readings"] = hwmon_readings
except Exception as e: sysfs["hwmon_temperature_readings"] = f"UNAVAILABLE: {type(e).__name__}: {e}"

# Node Exporter: scrape local loopback only; retain a compact summary, not the full scrape.
exporter = http_text("http://127.0.0.1:9100/metrics", timeout=5)
if exporter.get("ok"):
    exporter_summary = parse_prometheus_summary(exporter["body"])
    exporter_summary["http_status"] = exporter.get("status")
else:
    exporter_summary = exporter

# Detect the filesystem types actually mounted; this can reveal local, NFS, Lustre,
# BeeGFS, GPFS/Spectrum Scale or other shared-storage technologies if present.
mounts_text = results.get("findmnt_json", {}).get("stdout", "")

payload = {
    "identity": identity,
    "os_release": proc_files.get("/etc/os-release", ""),
    "commands": results,
    "services": services,
    "proc": proc_files,
    "sysfs": sysfs,
    "node_exporter_local": exporter_summary,
    "collection_notes": [
        "Read-only inventory; no sudo or configuration changes were performed.",
        "Unavailable commands and permission-limited paths are retained as collection gaps.",
        "A local Node Exporter scrape was attempted on 127.0.0.1:9100."
    ]
}
print(json.dumps(payload, ensure_ascii=False))
'''

## 3. Collect the headnode and compute-node inventories

The headnode is collected locally when its short hostname matches the configured node name. Other nodes are queried over SSH. SSH uses `BatchMode=yes`, so it will not ask for passwords or hang on an interactive prompt.

If remote access fails, first verify that you can run `ssh <node-address> hostname` from the headnode using the same account. Do not place passwords or private keys in this notebook.

In [3]:
def run_local_collector():
    proc = subprocess.run([sys.executable, "-c", COLLECTOR_SOURCE], text=True,
                          capture_output=True, timeout=90, check=False)
    if proc.returncode != 0:
        return {"reachable": False, "collection_error": f"local collector exit {proc.returncode}: {proc.stderr[-3000:]}"}
    try:
        data = json.loads(proc.stdout)
        data["reachable"] = True
        data["collection_method"] = "local Python subprocess"
        return data
    except Exception as e:
        return {"reachable": False, "collection_error": f"Could not parse local collector JSON: {e}",
                "stdout_tail": proc.stdout[-3000:], "stderr_tail": proc.stderr[-3000:]}


def run_remote_collector(address):
    ssh_cmd = [
        "ssh", "-T",
        "-o", "BatchMode=yes",
        "-o", f"ConnectTimeout={SSH_CONNECT_TIMEOUT_SECONDS}",
        "-o", "ConnectionAttempts=1",
        "-o", "ServerAliveInterval=4",
        "-o", "StrictHostKeyChecking=yes",
        f"{SSH_USER}@{address}",
        "python3 -"
    ]
    try:
        proc = subprocess.run(ssh_cmd, input=COLLECTOR_SOURCE, text=True,
                              capture_output=True, timeout=110, check=False)
        if proc.returncode != 0:
            return {"reachable": False, "collection_method": "SSH", "collection_error":
                    f"SSH/remote collector exit {proc.returncode}: {(proc.stderr or proc.stdout)[-3000:]}"}
        data = json.loads(proc.stdout)
        data["reachable"] = True
        data["collection_method"] = "SSH / remote Python collector"
        return data
    except subprocess.TimeoutExpired:
        return {"reachable": False, "collection_method": "SSH", "collection_error": "SSH collection timed out"}
    except Exception as e:
        return {"reachable": False, "collection_method": "SSH", "collection_error": f"{type(e).__name__}: {e}"}


inventory = {}
for node in CLUSTER_NODES:
    print(f"Collecting {node['name']} ({node['address']}) ...", end=" ")
    if node["name"] == local_short_name:
        record = run_local_collector()
    elif COLLECT_REMOTE_NODES:
        record = run_remote_collector(node["address"])
    else:
        record = {"reachable": False, "collection_error": "Remote collection disabled by configuration"}
    record["configured_name"] = node["name"]
    record["configured_address"] = node["address"]
    record["configured_role"] = node["role"]
    inventory[node["name"]] = record
    print("OK" if record.get("reachable") else "NOT COLLECTED")

reachable_count = sum(bool(v.get("reachable")) for v in inventory.values())
print(f"\nCollected {reachable_count} of {len(inventory)} configured nodes.")
if reachable_count == 0:
    raise RuntimeError("No node inventory was collected. Check the local hostname and SSH configuration.")


Collected 4 of 4 configured nodes.


## 4. Compare node identity, operating system, CPU and memory

This is the first consistency check. Differences are not automatically errors: the headnode may intentionally have a different role and hardware profile. Compute-node differences should be investigated before assuming a single shared model can be deployed identically on all nodes.

In [4]:
def command_text(record, key):
    return (record.get("commands", {}).get(key, {}) or {}).get("stdout", "")

def parse_lscpu(record):
    parsed = {}
    for line in command_text(record, "lscpu").splitlines():
        if ":" in line:
            k, v = line.split(":", 1)
            parsed[k.strip()] = v.strip()
    return parsed

def parse_meminfo_kib(record, key):
    raw = (record.get("proc", {}) or {}).get("/proc/meminfo", "")
    m = re.search(rf"^{re.escape(key)}:\s+(\d+)\s+kB", raw, re.M)
    return int(m.group(1)) if m else None

summary_rows = []
for node_name, record in inventory.items():
    if not record.get("reachable"):
        summary_rows.append({"configured_node": node_name, "reachable": False,
                             "error": record.get("collection_error", "not collected")})
        continue
    cpu = parse_lscpu(record)
    mem_kib = parse_meminfo_kib(record, "MemTotal")
    os_release = record.get("os_release", "")
    pretty = re.search(r'^PRETTY_NAME="?(.*?)"?$', os_release, re.M)
    summary_rows.append({
        "configured_node": node_name,
        "hostname_reported": record.get("identity", {}).get("hostname"),
        "reachable": True,
        "role": record.get("configured_role"),
        "os": pretty.group(1) if pretty else record.get("identity", {}).get("platform"),
        "kernel": record.get("identity", {}).get("os_release_kernel"),
        "architecture": record.get("identity", {}).get("architecture"),
        "cpu_model": cpu.get("Model name"),
        "logical_cpus": cpu.get("CPU(s)"),
        "sockets": cpu.get("Socket(s)"),
        "cores_per_socket": cpu.get("Core(s) per socket"),
        "threads_per_core": cpu.get("Thread(s) per core"),
        "numa_nodes": cpu.get("NUMA node(s)"),
        "memory_GiB": round(mem_kib / 1024 / 1024, 2) if mem_kib else None,
        "node_exporter_reachable": bool(record.get("node_exporter_local", {}).get("http_status")),
    })

try:
    import pandas as pd
    display(pd.DataFrame(summary_rows))
except Exception:
    for row in summary_rows:
        print(row)

print("\nInterpretation reminder: compare compute001–compute003 with each other separately from headnode.")

,configured_node,hostname_reported,reachable,role,os,kernel,architecture,cpu_model,logical_cpus,sockets,cores_per_socket,threads_per_core,numa_nodes,memory_GiB,node_exporter_reachable
0,headnode,headnode,True,headnode / management / monitoring,Ubuntu 22.04.5 LTS,5.15.0-1095-kvm,x86_64,AMD EPYC-Rome Processor,4,4,1,1,1,7.78,True
1,compute001,compute001,True,compute,Ubuntu 22.04.5 LTS,5.15.0-1095-kvm,x86_64,AMD EPYC-Rome Processor,8,8,1,1,1,15.64,True
2,compute002,compute002,True,compute,Ubuntu 22.04.5 LTS,5.15.0-1095-kvm,x86_64,AMD EPYC-Rome Processor,8,8,1,1,1,15.64,True
3,compute003,compute003,True,compute,Ubuntu 22.04.5 LTS,5.15.0-1095-kvm,x86_64,AMD EPYC-Rome Processor,8,8,1,1,1,15.64,True



Interpretation reminder: compare compute001–compute003 with each other separately from headnode.


## 5. Inspect each subsystem

The detailed command outputs are retained under `inventory`. The helper below prints selected sections without requiring root privileges. Review these sections to identify constraints and capabilities that may influence the monitoring and modelling design.

In [5]:
def show_section(node_name, section, key=None, max_chars=14000):
    rec = inventory.get(node_name, {})
    if not rec.get("reachable"):
        print(f"{node_name}: no inventory available — {rec.get('collection_error')}")
        return
    if section == "command":
        value = rec.get("commands", {}).get(key, {})
        print(f"[{node_name}] {key}\navailable={value.get('available')} returncode={value.get('returncode')}\n")
        print((value.get("stdout", "") or "")[:max_chars])
        if value.get("stderr"):
            print("\nSTDERR:", value["stderr"][:3000])
    elif section == "services":
        for name, result in rec.get("services", {}).items():
            print(f"--- {name} ---")
            print(result.get("stdout", "").strip() or result.get("stderr", "").strip())
    elif section == "proc":
        print((rec.get("proc", {}).get(key, "") or "")[:max_chars])
    elif section == "sysfs":
        print(json.dumps(rec.get("sysfs", {}).get(key), indent=2))
    elif section == "exporter":
        print(json.dumps(rec.get("node_exporter_local", {}), indent=2)[:max_chars])

# Example inspection calls. Change the node or section as needed.
show_section("compute001", "command", "lscpu")
show_section("compute001", "command", "lsblk_json")
show_section("compute001", "command", "df_bytes")
show_section("compute001", "command", "findmnt_json")
show_section("compute001", "command", "ip_addr_json")
show_section("compute001", "command", "slurm_sinfo")
show_section("compute001", "command", "slurm_nodes")

[compute001] lscpu
available=True returncode=0

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           48 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  8
On-line CPU(s) list:                     0-7
Vendor ID:                               AuthenticAMD
Model name:                              AMD EPYC-Rome Processor
CPU family:                              23
Model:                                   49
Thread(s) per core:                      1
Core(s) per socket:                      1
Socket(s):                               8
Stepping:                                0
BogoMIPS:                                5190.24
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 syscall nx mmxext fxsr_opt pdpe1gb rdtscp lm rep_good nopl cpui

## 6. HPC stack, scheduler, MPI and service consistency

For each node, check:

- Slurm client/daemon presence and whether the expected daemon is active for its role.
- Scheduler-visible CPU and memory resources versus operating-system-visible resources.
- MPI implementation and compiler wrapper consistency.
- Compiler and build-tool availability.
- Container runtime and accelerator tooling availability.
- Node Exporter service presence and direct scrape behaviour.

A command not installed on a particular node is not necessarily a fault. For example, controller-only services should not be expected on compute nodes.

In [6]:
for node_name, rec in inventory.items():
    print("\n" + "=" * 88)
    print(node_name, "—", "REACHABLE" if rec.get("reachable") else "NOT COLLECTED")
    if not rec.get("reachable"):
        print(rec.get("collection_error"))
        continue
    for key in ["slurm_version", "mpi_compiler", "mpi_launcher", "mpi_info_version",
                "gcc_version", "gfortran_version", "cmake_version", "apptainer_version",
                "singularity_version", "nvidia_smi_list", "rdma_devices"]:
        r = rec.get("commands", {}).get(key, {})
        first = (r.get("stdout") or r.get("stderr") or "").strip().splitlines()
        print(f"{key:24} {first[0][:160] if first else 'No output'}")
    print("\nService states:")
    for service, r in rec.get("services", {}).items():
        lines = [x for x in (r.get("stdout") or "").splitlines() if x.startswith(("LoadState=", "ActiveState=", "SubState=", "UnitFileState="))]
        print(f"  {service:30} {' '.join(lines) if lines else 'not reported'}")
    print("\nNode Exporter local summary:")
    exp = rec.get("node_exporter_local", {})
    print(f"  HTTP status: {exp.get('http_status', 'unavailable')}")
    print(f"  Scrape sample lines: {exp.get('sample_lines', 'n/a')}")
    print(f"  Metric families: {exp.get('metric_family_count', 'n/a')}")
    print(f"  Systemd unit-state series: {exp.get('systemd_unit_state_series', 'n/a')}")
    print(f"  Collectors reported: {len(exp.get('collector_success', {}))}")


headnode — REACHABLE
slurm_version            slurm-wlm 21.08.5
mpi_compiler             mpicc: Open MPI 4.1.4 (Language: C)
mpi_launcher             mpirun (Open MPI) 4.1.4
mpi_info_version         Open MPI v4.1.4
gcc_version              gcc (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
gfortran_version         GNU Fortran (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
cmake_version            cmake version 3.22.1
apptainer_version        command not found
singularity_version      command not found
nvidia_smi_list          command not found
rdma_devices             command not found

Service states:
  slurmctld                      LoadState=loaded ActiveState=active SubState=running UnitFileState=enabled
  slurmd                         LoadState=loaded ActiveState=inactive SubState=dead UnitFileState=enabled
  munge                          LoadState=loaded ActiveState=active SubState=running UnitFileState=enabled
  chrony                         LoadState=loaded ActiveState=active SubState=ru

## 7. Prometheus target health (headnode API)

This cell queries the Prometheus HTTP API without changing configuration. It reports the currently active targets for the `node-exporter` job and the Prometheus build version. If the API is not available at the configured URL, update `PROMETHEUS_URL` in Section 1 or record that this check could not be completed.

A target's current health is a point-in-time observation. It does not establish historical scrape continuity.

In [7]:
import urllib.request

def prometheus_api(path):
    url = PROMETHEUS_URL.rstrip("/") + path
    try:
        req = urllib.request.Request(url, headers={"Accept": "application/json", "User-Agent": "hpc-system-inventory/1.0"})
        with urllib.request.urlopen(req, timeout=HTTP_TIMEOUT_SECONDS) as response:
            return json.loads(response.read().decode("utf-8"))
    except Exception as e:
        return {"status": "unavailable", "error": f"{type(e).__name__}: {e}", "url": url}

prom_build = prometheus_api("/api/v1/status/buildinfo")
prom_targets = prometheus_api("/api/v1/targets?state=active")

print("Prometheus build information:")
if prom_build.get("status") == "success":
    print(json.dumps(prom_build.get("data", {}), indent=2))
else:
    print(json.dumps(prom_build, indent=2))

active_node_exporter_targets = []
if prom_targets.get("status") == "success":
    for target in prom_targets.get("data", {}).get("activeTargets", []):
        labels = target.get("labels", {})
        if labels.get("job") == "node-exporter":
            active_node_exporter_targets.append({
                "instance": labels.get("instance"),
                "job": labels.get("job"),
                "health": target.get("health"),
                "lastScrape": target.get("lastScrape"),
                "lastError": target.get("lastError"),
                "scrapeUrl": target.get("scrapeUrl"),
            })
    print("\nActive node-exporter targets:")
    try:
        display(pd.DataFrame(active_node_exporter_targets))
    except Exception:
        print(json.dumps(active_node_exporter_targets, indent=2))
else:
    print("\nPrometheus targets API unavailable:")
    print(json.dumps(prom_targets, indent=2))

Prometheus build information:
{
  "version": "3.15.0",
  "revision": "5241a27fe3c6983549fccc32f6e65917408c63cd",
  "branch": "HEAD",
  "buildUser": "root@4635dd007a60",
  "buildDate": "20260925-07:33:45",
  "goVersion": "go1.27.1"
}

Active node-exporter targets:


,instance,job,health,lastScrape,lastError,scrapeUrl
0,127.0.0.1:9100,node-exporter,up,2026-11-13T19:40:57.43785882Z,,http://127.0.0.1:9100/metrics
1,10.100.0.198:9100,node-exporter,up,2026-11-13T19:40:49.336785324Z,,http://10.100.0.198:9100/metrics
2,10.100.0.167:9100,node-exporter,up,2026-11-13T19:40:58.588928462Z,,http://10.100.0.167:9100/metrics
3,10.100.0.66:9100,node-exporter,up,2026-11-13T19:40:53.354411446Z,,http://10.100.0.66:9100/metrics


## 8. Telemetry and observability capability review

Use this section to inspect what the cluster can expose to a future monitoring pipeline:

- **CPU and memory pressure:** `/proc/pressure/*` (PSI), if supported and readable.
- **Thermal data:** `sensors`, `/sys/class/hwmon` and `/sys/class/thermal`.
- **Accelerators/interconnects:** NVIDIA/ROCm, RDMA and InfiniBand tools and sysfs inventory.
- **Storage:** filesystem types and mounts, block devices, disk statistics, NFS mounts.
- **Time:** Chrony tracking/sources. Time alignment is critical for joining metrics with events and labels.
- **Node Exporter:** version, metric families, successful collectors and available systemd unit-state series.

Do not conclude that a sensor or metric is unavailable cluster-wide based on one node. Compare all compute nodes and distinguish unsupported hardware from missing permissions or absent utilities.

In [8]:
for node_name, rec in inventory.items():
    print("\n" + "=" * 88)
    print(node_name)
    if not rec.get("reachable"):
        print(rec.get("collection_error")); continue
    print("PSI:")
    for path in ["/proc/pressure/cpu", "/proc/pressure/memory", "/proc/pressure/io"]:
        print(f"  {path}: {(rec.get('proc', {}).get(path, '') or '').strip().replace(chr(10), ' | ')}")
    print("Device/sensor inventory:")
    for key in ["hwmon_devices", "thermal_devices", "hwmon_temperature_readings", "infiniband_devices", "drm_devices", "nvme_devices", "cpu_frequency_governors"]:
        print(f"  {key}: {rec.get('sysfs', {}).get(key)}")
    print("NFS mounts/tool output:")
    print((rec.get("commands", {}).get("nfs_mounts", {}).get("stdout", "") or "").strip()[:4000] or "No nfsstat output")
    print("Chrony tracking:")
    print((rec.get("commands", {}).get("chrony_tracking", {}).get("stdout", "") or "").strip()[:2500] or "Chrony tracking not available")


headnode
PSI:
  /proc/pressure/cpu: some avg10=0.00 avg60=0.00 avg300=0.00 total=6688182271 | full avg10=0.00 avg60=0.00 avg300=0.00 total=0
  /proc/pressure/memory: some avg10=0.00 avg60=0.00 avg300=0.00 total=282097729 | full avg10=0.00 avg60=0.00 avg300=0.00 total=267502763
  /proc/pressure/io: some avg10=0.02 avg60=0.09 avg300=0.16 total=8761198592 | full avg10=0.02 avg60=0.09 avg300=0.13 total=8471526393
Device/sensor inventory:
  hwmon_devices: UNAVAILABLE: FileNotFoundError: [Errno 2] No such file or directory: '/sys/class/hwmon'
  thermal_devices: ['cooling_device0', 'cooling_device1', 'cooling_device2', 'cooling_device3']
  hwmon_temperature_readings: []
  infiniband_devices: UNAVAILABLE: FileNotFoundError: [Errno 2] No such file or directory: '/sys/class/infiniband'
  drm_devices: UNAVAILABLE: FileNotFoundError: [Errno 2] No such file or directory: '/sys/class/drm'
  nvme_devices: UNAVAILABLE: FileNotFoundError: [Errno 2] No such file or directory: '/sys/class/nvme'
  cpu_fr

## 9. Export the inventory for reproducibility

The notebook writes:

- `system_information_<timestamp>.json` — complete raw collection, including command outputs and collection errors.
- `system_information_summary_<timestamp>.csv` — compact node comparison table.
- `system_information_report_<timestamp>.md` — concise collection summary and reachability status.

Keep the JSON file with the project records. It is the source artifact for later environment comparisons. Review it before sharing outside the project because it contains internal hostnames, IP addresses, hardware identifiers and filesystem/network topology.

In [9]:
import csv
from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
json_path = OUTPUT_DIR / f"system_information_{stamp}.json"
csv_path = OUTPUT_DIR / f"system_information_summary_{stamp}.csv"
md_path = OUTPUT_DIR / f"system_information_report_{stamp}.md"

export_payload = {
    "inventory_metadata": {
        "created_at_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "created_at_local": datetime.now().astimezone().isoformat(timespec="seconds"),
        "collector_notebook": "system information.ipynb",
        "ssh_user": SSH_USER,
        "prometheus_url": PROMETHEUS_URL,
        "configured_nodes": CLUSTER_NODES,
        "collection_started_utc": collection_started_utc,
        "collection_started_local": collection_started_local,
        "read_only": True,
    },
    "nodes": inventory,
    "prometheus": {
        "build_info": prom_build,
        "node_exporter_targets": active_node_exporter_targets,
        "targets_api": prom_targets.get("status"),
    }
}
json_path.write_text(json.dumps(export_payload, indent=2, ensure_ascii=False), encoding="utf-8")

fieldnames = sorted({key for row in summary_rows for key in row.keys()})
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(summary_rows)

report_lines = [
    "# HPC System Information Collection Report",
    "",
    f"- Collection started (local): {collection_started_local}",
    f"- Collection started (UTC): {collection_started_utc}",
    f"- Collection completed (UTC): {datetime.now(timezone.utc).isoformat(timespec='seconds')}",
    f"- Nodes collected: {reachable_count}/{len(inventory)}",
    f"- Prometheus API status: {prom_targets.get('status', 'unknown')}",
    "",
    "## Node reachability",
    "",
    "| Configured node | Address | Role | Result | Reported hostname |",
    "|---|---|---|---|---|",
]
for node_name, rec in inventory.items():
    report_lines.append(
        f"| {node_name} | {rec.get('configured_address','')} | {rec.get('configured_role','')} | "
        f"{'Collected' if rec.get('reachable') else 'Not collected'} | "
        f"{rec.get('identity', {}).get('hostname', '')} |"
    )
report_lines += ["", "## Prometheus node-exporter targets", ""]
if active_node_exporter_targets:
    report_lines += ["| Instance | Health | Last scrape | Last error |", "|---|---|---|---|"]
    for target in active_node_exporter_targets:
        report_lines.append(f"| {target.get('instance')} | {target.get('health')} | {target.get('lastScrape')} | {target.get('lastError') or 'None'} |")
else:
    report_lines.append("No active node-exporter targets were returned, or the Prometheus targets API was unavailable.")
report_lines += ["", "## Collection limitations", "", 
                 "This is a read-only point-in-time inventory. Missing command output may indicate an absent utility, permissions, an unsupported device, or a service not running on that node. Investigate before interpreting missing values as missing hardware or missing telemetry.",
                 "", "## Files", "", f"- Raw JSON: `{json_path.name}`", f"- Summary CSV: `{csv_path.name}`"]
md_path.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved:")
print(json_path.resolve())
print(csv_path.resolve())
print(md_path.resolve())
print(f"\nInventory JSON size: {json_path.stat().st_size:,} bytes")

Saved:
/home/ubuntu/system_information_output/system_information_20261113T194100Z.json
/home/ubuntu/system_information_output/system_information_summary_20261113T194100Z.csv
/home/ubuntu/system_information_output/system_information_report_20261113T194100Z.md

Inventory JSON size: 323,394 bytes


## 10. How we will use the findings before modelling

Do not begin feature engineering until the inventory has been reviewed. Use the results to make explicit decisions about:

1. **Population and deployment scope:** whether the headnode should be modelled separately from compute nodes, and whether compute nodes are sufficiently homogeneous for a shared model.
2. **Hardware-aware interpretation:** CPU model/topology, NUMA layout, memory capacity, storage devices, interconnects and any accelerators.
3. **Metric feasibility:** which signals genuinely exist, at what layer they are collected, and whether all nodes expose them consistently.
4. **Telemetry design:** Node Exporter collector coverage, Prometheus target health, PSI and thermal sensor availability, time synchronisation and NFS visibility.
5. **Feature design:** which raw signals can support rolling means, rates of change, variability, pressure indicators and cross-metric interactions.
6. **Operational constraints:** available software, package versions, scheduler resource limits, storage capacity and where inference can run without disrupting HPC workloads.
7. **Historical-to-live compatibility:** compare the inventory and live metrics with the historical dataset's node identities, metric definitions, units and sampling cadence.

### Important research discipline

The inventory tells us what the environment can expose now. It does **not** tell us whether a metric predicts failure. Predictive usefulness must be established from the historical observations and validated against correctly defined event labels using chronological evaluation.

### Suggested review checklist

- [ ] All four configured nodes were collected, or each collection gap is explained.
- [ ] Compute-node CPU, memory and OS differences are understood.
- [ ] Slurm-visible resources are compared with OS-visible resources.
- [ ] MPI/compiler/library versions and differences are recorded.
- [ ] Storage and network topology are understood.
- [ ] PSI, temperature, fan and accelerator visibility are confirmed or explicitly marked unavailable.
- [ ] Node Exporter collector coverage and Prometheus target health are reviewed.
- [ ] Time synchronisation and clock consistency are checked.
- [ ] The inventory is compared with historical dataset metadata before modelling.